In [1]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-classifier")

X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=True)
X = X / 255.0
y = y.astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train,test_size=0.2,random_state=42,stratify=y_train)

2026/08/30 15:05:47 INFO mlflow.tracking.fluent: Experiment with name 'mnist-classifier' does not exist. Creating a new experiment.


In [2]:
def train_model(learning_rate_init=0.001,batch_size=64,hidden_layer_sizes=(128,),epochs=20):
    
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        learning_rate_init=learning_rate_init,
        batch_size=batch_size,
        solver = 'adam',
        max_iter=1,
        warm_start=True,
        random_state=42
    )
    
    train_losses = []
    val_accuracies = []

    for epoch in range(epochs):

        model.fit(X_train, y_train)

        train_loss = model.loss_

        val_predictions = model.predict(X_val)
        val_accuracy = accuracy_score(y_val, val_predictions)

        train_losses.append(train_loss)
        val_accuracies.append(val_accuracy)

        print(
            f"Epoch {epoch + 1:2d}: "
            f"train_loss={train_loss:.4f}, "
            f"val_accuracy={val_accuracy:.4f}"
        )

    return model, train_losses, val_accuracies

In [3]:
def train_and_log(learning_rate_init=0.001,batch_size=64,hidden_layer_sizes=(128,),epochs=20, run_name=None):
    with mlflow.start_run(run_name=run_name):
        model, train_losses, val_accuracies = train_model(learning_rate_init=learning_rate_init,batch_size=batch_size,hidden_layer_sizes=hidden_layer_sizes,epochs=epochs)
        mlflow.log_param("learning_rate_init", learning_rate_init)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("hidden_layer_sizes", hidden_layer_sizes)
        mlflow.log_param("epochs", epochs)

        for epoch in range(epochs):
            mlflow.log_metric("train_loss", train_losses[epoch],step=epoch)
            mlflow.log_metric("val_accuracy", val_accuracies[epoch], step=epoch)
        signature = infer_signature(X_train.iloc[:10], model.predict(X_train.iloc[:10]))
        mlflow.sklearn.log_model(model,"model",signature=signature,skops_trusted_types=["sklearn.neural_network._stochastic_optimizers.AdamOptimizer"])
        run_id = mlflow.active_run().info.run_id
    return run_id
baseline_run_id = train_and_log(0.001, 64, (128,), 20, run_name="mlp-baseline")

Epoch  1: train_loss=0.3417, val_accuracy=0.9436
Epoch  2: train_loss=0.1630, val_accuracy=0.9570
Epoch  3: train_loss=0.1130, val_accuracy=0.9639
Epoch  4: train_loss=0.0842, val_accuracy=0.9679
Epoch  5: train_loss=0.0652, val_accuracy=0.9693
Epoch  6: train_loss=0.0516, val_accuracy=0.9704
Epoch  7: train_loss=0.0412, val_accuracy=0.9707
Epoch  8: train_loss=0.0329, val_accuracy=0.9713
Epoch  9: train_loss=0.0262, val_accuracy=0.9720
Epoch 10: train_loss=0.0212, val_accuracy=0.9725
Epoch 11: train_loss=0.0170, val_accuracy=0.9723
Epoch 12: train_loss=0.0141, val_accuracy=0.9733
Epoch 13: train_loss=0.0112, val_accuracy=0.9740
Epoch 14: train_loss=0.0106, val_accuracy=0.9717
Epoch 15: train_loss=0.0096, val_accuracy=0.9746
Epoch 16: train_loss=0.0092, val_accuracy=0.9737
Epoch 17: train_loss=0.0098, val_accuracy=0.9744
Epoch 18: train_loss=0.0081, val_accuracy=0.9748
Epoch 19: train_loss=0.0074, val_accuracy=0.9740
Epoch 20: train_loss=0.0070, val_accuracy=0.9735


2026/08/30 15:06:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run mlp-baseline at: http://localhost:5000/#/experiments/1/runs/61f12774e168412b8d1a07c162e27b00
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [4]:
learning_rates = [0.001, 0.005]
batch_sizes = [32, 128, 256]

grid_run_ids = []
grid_results = []

for lr in learning_rates:
    for bs in batch_sizes:
        run_name = f"mlp-grid-lr{lr}-bs{bs}"
        rid = train_and_log(
            learning_rate_init=lr,
            batch_size=bs,
            hidden_layer_sizes=(128,),
            epochs=20,
            run_name=run_name
        )
        grid_run_ids.append(rid)
        grid_results.append({"learning_rate_init": lr, "batch_size": bs, "run_id": rid})

print("Grid run IDs:", grid_run_ids)

Epoch  1: train_loss=0.2960, val_accuracy=0.9515
Epoch  2: train_loss=0.1364, val_accuracy=0.9593
Epoch  3: train_loss=0.0920, val_accuracy=0.9655
Epoch  4: train_loss=0.0674, val_accuracy=0.9684
Epoch  5: train_loss=0.0513, val_accuracy=0.9688
Epoch  6: train_loss=0.0395, val_accuracy=0.9693
Epoch  7: train_loss=0.0309, val_accuracy=0.9669
Epoch  8: train_loss=0.0246, val_accuracy=0.9705
Epoch  9: train_loss=0.0201, val_accuracy=0.9701
Epoch 10: train_loss=0.0180, val_accuracy=0.9723
Epoch 11: train_loss=0.0161, val_accuracy=0.9735
Epoch 12: train_loss=0.0159, val_accuracy=0.9739
Epoch 13: train_loss=0.0145, val_accuracy=0.9736
Epoch 14: train_loss=0.0132, val_accuracy=0.9749
Epoch 15: train_loss=0.0127, val_accuracy=0.9723
Epoch 16: train_loss=0.0127, val_accuracy=0.9738
Epoch 17: train_loss=0.0117, val_accuracy=0.9718
Epoch 18: train_loss=0.0119, val_accuracy=0.9754
Epoch 19: train_loss=0.0115, val_accuracy=0.9744
Epoch 20: train_loss=0.0125, val_accuracy=0.9753


2026/08/30 15:07:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run mlp-grid-lr0.001-bs32 at: http://localhost:5000/#/experiments/1/runs/7ef8010acab54dacb016c5121a1b6731
🧪 View experiment at: http://localhost:5000/#/experiments/1
Epoch  1: train_loss=0.4085, val_accuracy=0.9356
Epoch  2: train_loss=0.1943, val_accuracy=0.9510
Epoch  3: train_loss=0.1411, val_accuracy=0.9585
Epoch  4: train_loss=0.1095, val_accuracy=0.9629
Epoch  5: train_loss=0.0878, val_accuracy=0.9654
Epoch  6: train_loss=0.0720, val_accuracy=0.9678
Epoch  7: train_loss=0.0597, val_accuracy=0.9687
Epoch  8: train_loss=0.0497, val_accuracy=0.9699
Epoch  9: train_loss=0.0415, val_accuracy=0.9708
Epoch 10: train_loss=0.0347, val_accuracy=0.9717
Epoch 11: train_loss=0.0290, val_accuracy=0.9730
Epoch 12: train_loss=0.0241, val_accuracy=0.9741
Epoch 13: train_loss=0.0200, val_accuracy=0.9736
Epoch 14: train_loss=0.0166, val_accuracy=0.9731
Epoch 15: train_loss=0.0139, val_accuracy=0.9727
Epoch 16: train_loss=0.0116, val_accuracy=0.9738
Epoch 17: train_loss=0.0096, val_accuracy=0

2026/08/30 15:08:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run mlp-grid-lr0.001-bs128 at: http://localhost:5000/#/experiments/1/runs/8d3d9d735106441d937a2d46a4ac8af4
🧪 View experiment at: http://localhost:5000/#/experiments/1
Epoch  1: train_loss=0.5109, val_accuracy=0.9263
Epoch  2: train_loss=0.2309, val_accuracy=0.9439
Epoch  3: train_loss=0.1729, val_accuracy=0.9532
Epoch  4: train_loss=0.1382, val_accuracy=0.9579
Epoch  5: train_loss=0.1141, val_accuracy=0.9613
Epoch  6: train_loss=0.0959, val_accuracy=0.9625
Epoch  7: train_loss=0.0818, val_accuracy=0.9647
Epoch  8: train_loss=0.0703, val_accuracy=0.9655
Epoch  9: train_loss=0.0608, val_accuracy=0.9668
Epoch 10: train_loss=0.0528, val_accuracy=0.9679
Epoch 11: train_loss=0.0461, val_accuracy=0.9693
Epoch 12: train_loss=0.0402, val_accuracy=0.9695
Epoch 13: train_loss=0.0352, val_accuracy=0.9699
Epoch 14: train_loss=0.0307, val_accuracy=0.9700
Epoch 15: train_loss=0.0270, val_accuracy=0.9708
Epoch 16: train_loss=0.0234, val_accuracy=0.9709
Epoch 17: train_loss=0.0204, val_accuracy=

...

🏃 View run mlp-grid-lr0.005-bs32 at: http://localhost:5000/#/experiments/1/runs/8fbf752eaf8845f0993a4546a10f7182
🧪 View experiment at: http://localhost:5000/#/experiments/1
Epoch  1: train_loss=0.2625, val_accuracy=0.9541
Epoch  2: train_loss=0.1246, val_accuracy=0.9650
Epoch  3: train_loss=0.0847, val_accuracy=0.9670
Epoch  4: train_loss=0.0631, val_accuracy=0.9666
Epoch  5: train_loss=0.0512, val_accuracy=0.9689
Epoch  6: train_loss=0.0448, val_accuracy=0.9694
Epoch  7: train_loss=0.0412, val_accuracy=0.9698
Epoch  8: train_loss=0.0358, val_accuracy=0.9686
Epoch  9: train_loss=0.0346, val_accuracy=0.9709
Epoch 10: train_loss=0.0320, val_accuracy=0.9745
Epoch 11: train_loss=0.0331, val_accuracy=0.9681
Epoch 12: train_loss=0.0314, val_accuracy=0.9719
Epoch 13: train_loss=0.0282, val_accuracy=0.9707
Epoch 14: train_loss=0.0274, val_accuracy=0.9718
Epoch 15: train_loss=0.0280, val_accuracy=0.9709
Epoch 16: train_loss=0.0295, val_accuracy=0.9718
Epoch 17: train_loss=0.0277, val_accuracy=0

2026/08/30 15:09:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run mlp-grid-lr0.005-bs128 at: http://localhost:5000/#/experiments/1/runs/344fcfc0165a49a3a70186af5b03a1af
🧪 View experiment at: http://localhost:5000/#/experiments/1
Epoch  1: train_loss=0.3112, val_accuracy=0.9511
Epoch  2: train_loss=0.1382, val_accuracy=0.9604
Epoch  3: train_loss=0.0917, val_accuracy=0.9646
Epoch  4: train_loss=0.0676, val_accuracy=0.9665
Epoch  5: train_loss=0.0508, val_accuracy=0.9693
Epoch  6: train_loss=0.0419, val_accuracy=0.9682
Epoch  7: train_loss=0.0368, val_accuracy=0.9679
Epoch  8: train_loss=0.0310, val_accuracy=0.9643
Epoch  9: train_loss=0.0309, val_accuracy=0.9687
Epoch 10: train_loss=0.0261, val_accuracy=0.9705
Epoch 11: train_loss=0.0263, val_accuracy=0.9677
Epoch 12: train_loss=0.0209, val_accuracy=0.9723
Epoch 13: train_loss=0.0198, val_accuracy=0.9726
Epoch 14: train_loss=0.0230, val_accuracy=0.9651
Epoch 15: train_loss=0.0194, val_accuracy=0.9712
Epoch 16: train_loss=0.0179, val_accuracy=0.9715
Epoch 17: train_loss=0.0177, val_accuracy=

2026/08/30 15:09:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run mlp-grid-lr0.005-bs256 at: http://localhost:5000/#/experiments/1/runs/db25132230a74ba1a1beb8addad1c1bf
🧪 View experiment at: http://localhost:5000/#/experiments/1
Grid run IDs: ['7ef8010acab54dacb016c5121a1b6731', '8d3d9d735106441d937a2d46a4ac8af4', '2b45112b9e6a4610b097cf3d93ac56ea', '8fbf752eaf8845f0993a4546a10f7182', '344fcfc0165a49a3a70186af5b03a1af', 'db25132230a74ba1a1beb8addad1c1bf']


In [5]:
runs_df = mlflow.search_runs(experiment_names=["mnist-classifier"])
best_run = runs_df.sort_values("metrics.val_accuracy", ascending=False).iloc[0]
print(best_run[["run_id", "params.learning_rate_init", "params.batch_size", "metrics.val_accuracy"]])

run_id                       7ef8010acab54dacb016c5121a1b6731
params.learning_rate_init                               0.001
params.batch_size                                          32
metrics.val_accuracy                                 0.975268
Name: 5, dtype: object
